In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report

)

from imblearn.over_sampling import SMOTE

import joblib

In [ ]:
# LOAD DATASET

df = pd.read_csv(
    "../data/explainable_jobs.csv"
)

print(df.shape)

(4661, 58)


In [ ]:
# LEAKAGE SAFE FEATURES

features = [

    "keyword_score",

    "skill_count",

    "description_length",

    "readability_score",

    "missing_info_score",

    "trust_score",

    "contact_risk",

    "website_available",

    "email_available",

    "salary_available",

    "short_description_flag",

    "skills_missing",

    "company_about_missing",

    "deadline_pressure_score"

]

X = df[features]

y = df["fraud_label"]

print(X.shape)

(4661, 14)


In [ ]:
# ENCODE LABELS

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print(
    encoder.classes_
)

['GENUINE' 'SCAM']


In [ ]:
# =====================================================
# SPLIT DATA
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    stratify=y,

    random_state=42

)

print(X_train.shape)
print(X_test.shape)

(3728, 14)
(933, 14)


In [ ]:
# =====================================================
# SMOTE
# =====================================================

smote = SMOTE(
    random_state=42
)

X_train_resampled, y_train_resampled = (

    smote.fit_resample(

        X_train,

        y_train

    )

)

print(
    X_train_resampled.shape
)

(6782, 14)


In [ ]:
# =====================================================
# SCALING
# =====================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(

    X_train_resampled

)

X_test_scaled = scaler.transform(

    X_test

)

In [ ]:
# =====================================================
# SCALING
# =====================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(

    X_train_resampled

)

X_test_scaled = scaler.transform(

    X_test

)

In [ ]:
# =====================================================
# MODELS
# =====================================================

models = {

    "Logistic Regression":

        LogisticRegression(

            max_iter=1000,

            class_weight="balanced"

        ),

    "Random Forest":

        RandomForestClassifier(

            n_estimators=300,

            max_depth=8,

            min_samples_leaf=5,

            class_weight="balanced",

            random_state=42

        ),

    "XGBoost":

        XGBClassifier(

            n_estimators=300,

            max_depth=4,

            learning_rate=0.05,

            subsample=0.8,

            colsample_bytree=0.8,

            random_state=42

        )

}

In [ ]:
# =====================================================
# TRAIN MODELS
# =====================================================

results = []

for name, model in models.items():

    print("\n" + "="*50)

    print(name)

    if name == "Logistic Regression":

        model.fit(

            X_train_scaled,

            y_train_resampled

        )

        preds = model.predict(

            X_test_scaled

        )

    else:

        model.fit(

            X_train_resampled,

            y_train_resampled

        )

        preds = model.predict(

            X_test

        )

    accuracy = accuracy_score(

        y_test,

        preds

    )

    precision = precision_score(

        y_test,

        preds,

        zero_division=0

    )

    recall = recall_score(

        y_test,

        preds,

        zero_division=0

    )

    f1 = f1_score(

        y_test,

        preds,

        zero_division=0

    )

    results.append([

        name,

        accuracy,

        precision,

        recall,

        f1

    ])

    print(

        classification_report(

            y_test,

            preds

        )

    )


Logistic Regression
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       849
           1       1.00      0.99      0.99        84

    accuracy                           1.00       933
   macro avg       1.00      0.99      1.00       933
weighted avg       1.00      1.00      1.00       933


Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       849
           1       0.95      0.99      0.97        84

    accuracy                           0.99       933
   macro avg       0.98      0.99      0.98       933
weighted avg       0.99      0.99      0.99       933


XGBoost
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       849
           1       0.97      0.99      0.98        84

    accuracy                           1.00       933
   macro avg       0.98      0.99      0.99       933
weighted avg       1.00      

In [ ]:
# =====================================================
# MODEL COMPARISON
# =====================================================

results_df = pd.DataFrame(

    results,

    columns=[

        "Model",

        "Accuracy",

        "Precision",

        "Recall",

        "F1 Score"

    ]

)

results_df.sort_values(

    by="F1 Score",

    ascending=False

)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.998928,1.000000,0.988095,0.994012
2,XGBoost,0.995713,0.965116,0.988095,0.976471
1,Random Forest,0.994641,0.954023,0.988095,0.970760


In [ ]:
# =====================================================
# FINAL MODEL
# =====================================================

best_model = XGBClassifier(

    n_estimators=300,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42

)

best_model.fit(

    X_train_resampled,

    y_train_resampled

)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# =====================================================
# FEATURE IMPORTANCE
# =====================================================

importance_df = pd.DataFrame({

    "Feature": features,

    "Importance":

        best_model.feature_importances_

})

importance_df.sort_values(

    by="Importance",

    ascending=False

)

,Feature,Importance
4,missing_info_score,0.298270
5,trust_score,0.280300
0,keyword_score,0.227954
6,contact_risk,0.093726
11,skills_missing,0.077426
2,description_length,0.010773
3,readability_score,0.004481
1,skill_count,0.002763
10,short_description_flag,0.002759
7,website_available,0.000927


In [ ]:
# =====================================================
# SAVE MODEL
# =====================================================

joblib.dump(

    best_model,

    "../models/xgboost.pkl"

)

joblib.dump(

    scaler,

    "../models/scaler.pkl"

)

print(
    "Model Saved Successfully"
)

Model Saved Successfully


In [ ]:
# =====================================================
# SAVE COMPARISON TABLE
# =====================================================

results_df.to_csv(

    "../data/model_comparison.csv",

    index=False

)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.781350,0.211538,0.523810,0.301370
1,Random Forest,0.771704,0.205479,0.535714,0.297030
2,XGBoost,0.841372,0.280822,0.488095,0.356522


In [ ]:
features

['description_length',
 'readability_score',
 'skill_count',
 'website_available',
 'email_available',
 'salary_available',
 'skills_missing',
 'company_about_missing',
 'deadline_pressure_score']

In [ ]:
importance_df.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
6,skills_missing,0.379080
0,description_length,0.179061
2,skill_count,0.120064
3,website_available,0.105581
4,email_available,0.074405
1,readability_score,0.065839
8,deadline_pressure_score,0.045065
7,company_about_missing,0.030904
5,salary_available,0.000000
